# TEST HDEMUCS SEPARATION MODEL

### Imports

In [1]:
import torch
import os
import yaml
import soundfile as sf
from demucs.hdemucs import HDemucs
from demucs.apply import apply_model

import ipywidgets as widgets
from IPython.display import display, Audio, clear_output

### <span style="color: yellow">Put your checkpoint here<span>

In [ ]:
# checkpoint_path = './lightning_logs/PyRoom_PerceptualLoss_HDEMUCS_S2S_multi_channels=48/version_3/checkpoints/best_epoch=110.ckpt'
# checkpoint_path = '../HDemucs_Exps/lightning_logs/PyRoom_LossL1_HDEMUCS_S2S_multi_channels=48/version_0/checkpoints/best_epoch=180.ckpt'
# checkpoint_path = '../HDemucs_Exps/lightning_logs/PyRoom_LossL1_HDEMUCScac_S2S_multi_channels=48/version_0/checkpoints/best_epoch=41.ckpt'
# checkpoint_path = '../HDemucs_Exps/lightning_logs/PyRoom_AuralossNEW_HDEMUCS_S2S_multi_channels=48/version_1/checkpoints/best_epoch=191.ckpt'
# checkpoint_path = '../HDemucs_Exps/lightning_logs/PyRoom_AuralossNEW_HDEMUCScac_S2S_multi_channels=48/version_0/checkpoints/best_epoch=16.ckpt'
#checkpoint_path = './checkpoints/best_epoch=186.ckpt'

checkpoint_path = '/mnt/home/mdbm0009/HDemucs_ExpSpheres/checkpoints/best_epoch=155.ckpt'
#checkpoint_path = './lightning_logs/Spheres_HDEMUCS_loss=l1_channels=48_perm=False_cac=False/version_6/checkpoints/best_epoch=186.ckpt'

### <span style="color: yellow">Put input mixture here<span>

In [13]:
# audio_path = '/mnt/REPERTORIUM/results/HDEMUCS_S2S_Auraloss_permutation/single_acoustics/spheres_strings/Mozart_445'
# audio_path = os.path.join('/mnt/home/pcabanas/BBDD/SynthSOD/PyRoomDataset_single_acoustics/test/same_acoustics_and_layout/rsno_centre_cardioids_rt60_1200', 'symphony_6_1_orch')
# audio_path = '/mnt/REPERTORIUM/BBDD/Sequenza/ID_128/Mozart_Duo_in_B'
# audio_path = '/mnt/home/pcabanas/BBDD/Pruebas_voz'

#audio_path = '/mnt/share/pcabanas/SPHERES/Formatted/Tchaikovsky/'

audio_path = '/mnt/home/mdbm0009/DataBase_Demucs_Reorganizada/validation/combinacion11/eb19249ef4416f33a107ec6f60dddcec/inputs'
audio_path = '/mnt/home/mdbm0009/DataBase_Demucs_Reorganizada/validation/combinacion873/fa1a56cb48b4df1760cf2783befcc95a/inputs6'

audio_name = 'mixture.wav'

### Load model / checkpoint

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# create an instance of HDemucs
with open('./conf.yaml','r') as f:
    conf = yaml.safe_load(f)
S = 12
confdemucs = {
    'sources': [str(i) for i in range(conf['separator_conf']['n_srcs'])],  # ['0', '1', ..., '11']
    'audio_channels': conf['separator_conf']['n_imics'],  # 12 micrófonos
    'cac': conf['hdemucs_conf']['cac'],                  # false
    'samplerate': conf['hdemucs_conf']['samplerate'],    # 48000
    'channels': conf['hdemucs_conf']['channels'],        # 6 canales internos
    'segment': conf['dataset']['chunk_duration'],        # 4 segundos
}
model = HDemucs(**confdemucs).to(device)

# load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)

# clean state_dict
state_dict = {
    k.replace("model.", ""): v for k, v in checkpoint["state_dict"].items()
}
state_dict = {
    k: v for k, v in state_dict.items() if not k.startswith("auralossnew.")
}

# load state_dict into the model
model.load_state_dict(state_dict)
model.eval()

clear_output(wait=True)

/tmp/ipykernel_861345/3577464675.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=device)


### Read audio properties

In [15]:
mix_path = os.path.join(audio_path, audio_name)
info = sf.info(mix_path)
fs = info.samplerate
duration = info.frames / fs

start_slider = widgets.FloatSlider(min=0, max=duration, step=0.1, description='Start (s)')
final_slider = widgets.FloatSlider(min=0, max=duration, step=0.1, description='End (s)', value=duration)
display(widgets.VBox([start_slider, final_slider]))

### <span style="color: yellow">Select audio range (use sliders above)<span>

In [16]:
waveform, _ = sf.read(
                mix_path,
                start = int(start_slider.value * fs),
                stop = int(final_slider.value * fs),
                fill_value = 0.0,
                dtype = 'float32'
                )
waveform = torch.tensor(waveform, dtype=torch.float32).to(device)
waveform = waveform.permute(1, 0)[None] # permute to (B, C, T)

# fill with zeros if the number of audio channels does not match with model
B, C, T = waveform.shape
if C < model.audio_channels:
    pad_channels = model.audio_channels - C
    padding = torch.zeros(B, pad_channels, T, device=waveform.device, dtype=waveform.dtype)
    waveform = torch.cat([waveform, padding], dim=1)

for c in range(0,C):
    print(f'Channel {c}:')
    display(Audio(data=waveform[0,c,:].cpu().numpy(), rate=fs))

Channel 0:


Channel 1:


Channel 2:


Channel 3:


Channel 4:


Channel 5:


### Separate audio

In [17]:
with torch.no_grad():
    separated = apply_model(model, waveform, shifts=0)  # output is (1, S, T)

# normalize the output (sometimes the output is above 1.0)
separated = separated / separated.abs().max()

for c in range(0,model.audio_channels):
    print(f'Channel {c}:')
    display(Audio(data=separated[0,c,:].cpu().numpy(), rate=fs))

Channel 0:


Channel 1:


Channel 2:


Channel 3:


Channel 4:


...

Channel 7:


Channel 8:


Channel 9:


Channel 10:


Channel 11:


In [10]:
# Save separated and wavewform signals as wav files in forder ./wav
output_folder = './wav'
os.makedirs(output_folder, exist_ok=True)
sf.write(os.path.join(output_folder, 'mixture.wav'), waveform[0,:C,:].cpu().numpy().T, fs)
for s in range(separated.shape[1]):
    sf.write(os.path.join(output_folder, f'source_{s}.wav'), separated[0,s,:].cpu().numpy(), fs)
